In [7]:
import jax
import jax.numpy as jnp
from jax import jit, grad, vmap
from functools import partial
print(jax.devices())

[CpuDevice(id=0)]


# Some Basic Modifications in JAX compared with numpy

In [ ]:
key = jax.random.PRNGKey(0)
key1, key2 = jax.random.split(key)

In [5]:
print(key1, key2, key)

[1797259609 2579123966] [ 928981903 3453687069] [0 0]


In [8]:
A = jnp.array([[1.0, 2.0], [3.0, 4.0]])
# A[0, 0] = 5.0  # Outputs ERROR: JAX arrays are immutable
A_new = A.at[0, 0].set(5.0)  # returns a new array

In [ ]:
# === JIT (Just In Time) compilation ===
def slow_fn(x):
    for _ in range(10):
        x = x @ x  # matrix power
    return x

fast_fn = jit(slow_fn)

# Warm up (traces + compiles on first call)
_ = fast_fn(jnp.eye(2))

# Benchmark
%timeit slow_fn(jnp.eye(100)).block_until_ready()
%timeit fast_fn(jnp.eye(100)).block_until_ready()

595 μs ± 14.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
416 μs ± 15.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
# === Automatic differentiation ===
# Scalar function: gradient is straightforward
def f(x):
    return jnp.sum(x ** 2)

grad_f = grad(f)
print(grad_f(jnp.array([3.0, 4.0])))  # [6.0, 8.0]

# Matrix-valued: need to differentiate scalar-valued objectives
# Gradient of log-determinant (important for Gaussian likelihoods)
def log_det(L):
    """L is a lower triangular Cholesky factor."""
    return 2.0 * jnp.sum(jnp.log(jnp.diag(L)))

L = jnp.array([[2.0, 0.0], [1.0, 3.0]])
print(grad(log_det)(L))
# Analytically: d/dL log|LL^T| = 2 * L^{-T} for diagonal, off-diag entries are 0

# === VMAP: Automatic vectorization ===
def matvec(A, x):
    return A @ x

# Apply to a batch of vectors simultaneously
batch_matvec = vmap(matvec, in_axes=(None, 0))  # A fixed, x batched along axis 0

A = jnp.eye(3)
X = jax.random.normal(jax.random.PRNGKey(0), (100, 3))
result = batch_matvec(A, X)  # shape: (100, 3)